<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c42-header.png" alt="Nextia Learning · Vision AI: See, Count and Inspect" width="100%">

# Solution: CLIP zero-shot

**[CLIP zero-shot](https://learning.nextia-ai.com/courses/vision/m03/clip-zero-shot/)** · Vision AI: See, Count and Inspect · Module 3, lesson 2 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** classify catalogue photos with text prompts and no labelled photos, read the probabilities of one photo, compare three prompt sets, and choose a class name on the validation photos.

| | |
|---|---|
| **Time** | About 30 minutes |
| **Needs** | An internet connection for the setup cells and the CLIP model (605 MB, downloaded once). No account and no key. |
| **You should know** | Embeddings and cosine similarity, from [Image embeddings](https://learning.nextia-ai.com/courses/vision/m03/image-embeddings/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1, Pillow 13.0.0, scikit-learn 1.9.1 and matplotlib 3.11.2 (PyTorch 2.15.0 and timm 1.0.31, open_clip 4.2.1 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/vision/m03/clip-zero-shot/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C42/M03-L02-clip-zero-shot/clip-zero-shot-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, pandas 3.1.1, pillow 13.0.0, scikit-learn 1.9.1, scikit-image 0.26.0, torch 2.15.0, torchvision 0.29.2, timm 1.0.31, open-clip-torch 4.2.1, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time. PyTorch is large (about 200 MB to download): the first run of this cell can take 2 minutes.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 pillow==13.0.0 scikit-learn==1.9.1 scikit-image==0.26.0 matplotlib==3.11.2 torch==2.15.0 torchvision==0.29.2 timm==1.0.31 open-clip-torch==4.2.1
import warnings; warnings.filterwarnings("ignore")
import numpy; print("numpy", numpy.__version__)
import pandas; print("pandas", pandas.__version__)
import PIL; print("pillow", PIL.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)
import skimage; print("scikit-image", skimage.__version__)
import torch; print("torch", torch.__version__)
import torchvision; print("torchvision", torchvision.__version__)
import timm; print("timm", timm.__version__)
import open_clip; print("open-clip-torch", open_clip.__version__)

## Setup: get the Larkfield images

Run the next cell. It downloads the image sets that this lesson needs (about 19 MB, as zip files) into a folder `larkfield-images`, with the course's helper `compare.py`, checks each file's checksum and unpacks it. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-images` (in Colab: the **Files** panel on the left), then run the cell again.

Where the photos come from: the catalogue photos are cut from Open Images photos (CC BY 2.0: the author of each photo is in `data/catalogue/manifest.csv`), the returned candles are photos of the VisA inspection set (CC BY 4.0), and the pallets are drawn by our code. Read `data/dataset-card.md` for the details.

In [ ]:
# Setup: download the image sets into larkfield-images/, check them and unpack them.
import hashlib, os, shutil, subprocess, urllib.request, zipfile
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C42/"
FILES = {
    "archives/catalogue.zip": "83752b3ce15a1b99f3b9201cc979769d7abe39633754a84865d7e60f0b3d7814",
    "archives/extras.zip": "f121137bc765866c25929a4233c557ba72b64eea0c806070cc04a86c28ec27ec",
    "compare.py": "e9794acacbce00c240906cc8b3e2256b8505ff4a8074b7507a8b159cd39b4826",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-images").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-images and run again.")
    if name.endswith(".zip"):
        with zipfile.ZipFile(path) as z:
            z.extractall(PROJECT / "data")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
n = sum(1 for p in Path("data").rglob("*") if p.is_file())
print(f"Ready: {n} files in larkfield-images/data.")

## Setup: helpers

The next cell loads NumPy as `np`, pandas as `pd`, Pillow's `Image`, matplotlib, and the course helper as `cp` (the comparison table). It defines `show(paths)`, which draws photos in a row, and the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: images, the comparison table, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import compare as cp
warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
CLASSES = ["bowl", "candle", "chair", "cushion", "lamp", "mug", "plant_pot", "vase"]

def show(paths, titles=None, size=2.2):
    """Draw photos in a row (at most 8)."""
    paths = list(paths)[:8]
    fig, axes = plt.subplots(1, len(paths), figsize=(size * len(paths), size))
    for i, (ax, p) in enumerate(zip(np.atleast_1d(axes), paths)):
        ax.imshow(Image.open(p)); ax.axis("off")
        if titles:
            ax.set_title(str(titles[i]), fontsize=8)
    plt.show()

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Load CLIP

CLIP has two parts: an **image encoder** and a **text encoder**. Both turn their input into 512 numbers in the same space. The course team already ran the image encoder on every catalogue photo (`data/embeddings/clip.npz`), so this notebook needs only the text encoder for the prompts. The next cell downloads the model once (605 MB). The first time, it takes one or two minutes.

In [ ]:
import torch, open_clip
from huggingface_hub import hf_hub_download
torch.set_num_threads(4)
# CLIP ViT-B/32 trained by LAION (MIT licence), at a fixed revision so that everyone gets the same numbers
weights = hf_hub_download("laion/CLIP-ViT-B-32-laion2B-s34B-b79K", "open_clip_model.safetensors",
                          revision="1a25a446712ba5ee05982a381eed697ef9b435cf")
model, _, preprocess = open_clip.create_model_and_transforms("ViT-B-32", pretrained=weights)
model.eval()
tokenizer = open_clip.get_tokenizer("ViT-B-32")

def text_matrix(names, templates):
    """One row of 512 numbers for each class: the mean of its prompts, scaled to length 1."""
    rows = []
    with torch.no_grad():
        for name in names:
            e = model.encode_text(tokenizer([t.format(name) for t in templates]))
            e = e / e.norm(dim=-1, keepdim=True)
            rows.append(e.mean(0))
        T = torch.stack(rows)
        return (T / T.norm(dim=-1, keepdim=True)).numpy()

def zero_shot(X, names, templates):
    """For each photo, the class whose text has the highest cosine similarity."""
    return [CLASSES[i] for i in (X @ text_matrix(names, templates).T).argmax(1)]

clip = np.load("data/embeddings/clip.npz")
catalogue = pd.read_csv("data/catalogue/manifest.csv")
val = catalogue[catalogue["split"] == "val"].reset_index(drop=True)
test = catalogue[catalogue["split"] == "test"].reset_index(drop=True)

def embeddings(ids):
    X = np.stack([clip[i] for i in ids]).astype(np.float32)
    return X / np.linalg.norm(X, axis=1, keepdims=True)

Xval, Xtest = embeddings(val["image_id"]), embeddings(test["image_id"])
NAMES = [c.replace("_", " ") for c in CLASSES]  # "plant pot", not "plant_pot"
EIGHT = ["a photo of a {}.", "a product photo of a {}.", "a close-up photo of a {}.", "a photo of a {} for sale.",
         "a {} in a home.", "a photo of the {}.", "a cropped photo of a {}.", "a bright photo of a {}."]
PROMPT_SETS = {"class name only": ["{}"], "a photo of a {}": ["a photo of a {}."], "eight templates": EIGHT}
print(Xval.shape, Xtest.shape)

> **Check.** You should see `(120, 512) (160, 512)`.

The stored embeddings are the model's own. The next cell embeds one photo again, live, and compares it with the stored one. A cosine of 1.000 means the same direction.

In [ ]:
i = test["image_id"].tolist().index("P-5de837d3")
with torch.no_grad():
    live = model.encode_image(preprocess(Image.open("data/" + test["file"][i]).convert("RGB"))[None]).numpy()[0]
live = live / np.linalg.norm(live)
print("cosine of the live and the stored embedding:", round(float(live @ Xtest[i]), 3))

> **Check.** You should see 1.0 (the stored numbers are rounded to float16).

## 2. One photo against eight prompts

Two plant pots from the test photos. For each one, the cell computes the cosine similarity with the eight prompts "a photo of a {class}.". CLIP multiplies the cosines by 100 and turns them into probabilities with the **softmax**.

> **Predict.** Photo 2 is a bucket of sunflowers. Which two classes will be close?

In [ ]:
T = text_matrix(NAMES, ["a photo of a {}."])
for pid in ["P-5de837d3", "P-533ffe5f"]:
    i = test["image_id"].tolist().index(pid)
    cos = Xtest[i] @ T.T
    z = 100 * cos
    prob = np.exp(z - z.max()) / np.exp(z - z.max()).sum()
    show(["data/" + test["file"][i]], [pid], size=2.5)
    print(pd.DataFrame({"cosine": cos.round(3), "probability": prob.round(3)}, index=NAMES).sort_values("cosine", ascending=False).head(4))

> **Check.** Photo 1 gets plant pot with a probability above 0.99. Photo 2 is a close call: vase 0.563 against plant pot 0.434, so CLIP says vase.

## 3. Three prompt sets

The same eight classes, three ways to write the prompts. A prompt set with several templates averages the text embeddings of each class.

In [ ]:
rows = {}
for name, templates in PROMPT_SETS.items():
    predicted = zero_shot(Xtest, NAMES, templates)
    rows[name] = cp.classification_scores(list(test["label"]), predicted, CLASSES)
pd.DataFrame({name: {"accuracy": s["accuracy"], "plant pot recall": s["per_class"]["plant_pot"]["recall"],
                     "worst class": s["worst_class"]} for name, s in rows.items()}).T

> **Check.** Accuracy 0.9, 0.906 and 0.925. "a photo of a plant pot" does **not** beat "plant pot" here: the plant pot recall falls from 0.90 to 0.80.

## 4. Your turn: another name for one class

Open Images calls the plant pot class "Flowerpot". Does CLIP know that name better? Choose on the **validation** photos, not on the test.

> **Your turn.** Copy `NAMES`, replace the name of the plant pot class by `"flowerpot"`, and classify the validation photos `Xval` with the eight templates `EIGHT`. Put the share of right answers, rounded to 3 decimals, in `flowerpot_val`.

In [ ]:
names = NAMES.copy()
names[CLASSES.index("plant_pot")] = "flowerpot"
flowerpot_val = round(float(np.mean(np.array(zero_shot(Xval, names, EIGHT)) == val["label"].values)), 3)
print(flowerpot_val)

In [ ]:
expect('flowerpot_val', flowerpot_val, 0.967, tolerance=0.0005)

Now compare with the name "plant pot" on the same validation photos:

In [ ]:
plant_pot_val = round(float(np.mean(np.array(zero_shot(Xval, NAMES, EIGHT)) == val["label"].values)), 3)
print("plant pot:", plant_pot_val, " flowerpot:", flowerpot_val if flowerpot_val is not ... else "not yet")

> **Check.** Both names give 0.967 on the 120 validation photos: 116 right. The validation set cannot choose between them. On the test set, "flowerpot" would give 0.938. But if you choose the name **because** of the test score, the test score is no longer honest. Keep "plant pot", or try "flowerpot" on new labelled photos first.

## 5. Failure case: one prompt for "damaged"

CLIP knows everyday words well, and small defects badly. The comparison table has a row for the prompts "a perfect, undamaged tea light candle" against "a damaged tea light candle with a defect":

In [ ]:
table = cp.Table.load("data/comparison_table.json")
table.show("inspect", columns=("method", "setting", "value", "worst", "worst_value"))

> **Check.** The CLIP prompt costs about 1,013 dollars a day on the returns test: better than no inspection (2,229), much worse than the best inspection method of Module 6. Zero-shot works for **what** a photo shows, not for small details.

## Next

The [lesson page](https://learning.nextia-ai.com/courses/vision/m03/clip-zero-shot/) has the diagrams and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Linear probes and few-shot](https://learning.nextia-ai.com/courses/vision/m03/linear-probes-and-few-shot/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/vision/m03/clip-zero-shot/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The photos are open photos (each with its author and licence in the manifests) or images made by our code; no photo shows a real person.